In [5]:
import geopandas as gpd
from matplotlib import pyplot as plt
import pandas as pd
import networkx as nx
import h3
from networkx.algorithms.community import louvain_communities
import igraph as ig
import leidenalg
import networkx as nx
import igraph as ig
import leidenalg
import geopandas as gpd
import pandas as pd
import h3
from tqdm import tqdm

import numpy as np
import random
seed = 505
random.seed(seed)        # or any integer
np.random.seed(seed)

import libpysal
import spopt
from spopt.region import RegionKMeansHeuristic
input_dir = "../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/"
retail_centres = gpd.read_parquet(input_dir + "3_afterfill.parquet")

In [19]:
import networkx as nx
import h3
import numpy as np
from tqdm import tqdm

threshold = 4  # POI count threshold for keeping edges
retail_centres = retail_centres.copy()

region_labels = {}

for network_number in tqdm(retail_centres['network_number'].unique()):
    rc = retail_centres[retail_centres['network_number'] == network_number].copy()
    rc = rc.dropna(subset=['foursquare_count'])
    
    if rc.empty:
        continue

    h3_list = rc['h3_11'].tolist()
    foursquare_count = dict(zip(rc['h3_11'], rc['foursquare_count']))
    h3_index = set(h3_list)

    G = nx.Graph()
    G.add_nodes_from(h3_list)

    for h in h3_list:
        for n in h3.grid_ring(h, 1):
            if n in h3_index:
                if (
                    foursquare_count.get(h, 0) >= threshold and 
                    foursquare_count.get(n, 0) >= threshold
                ):
                    G.add_edge(h, n)

    components = list(nx.connected_components(G))

    for region_id, component in enumerate(components):
        for h in component:
            region_labels[h] = region_id

retail_centres[f'region_graph_thresh{threshold}'] = retail_centres['h3_11'].map(region_labels)
retail_centres[f'region_graph_thresh{threshold}'] = (
    retail_centres.groupby('network_number')[f'region_graph_thresh{threshold}']
    .transform(lambda x: pd.factorize(x)[0])
)

# Save to disk
retail_centres.to_parquet(input_dir + f"4_region_graph_thresh{threshold}.parquet", index=False)


  0%|          | 0/12044 [00:00<?, ?it/s]

100%|██████████| 12044/12044 [00:18<00:00, 657.53it/s]


In [21]:
# Load data
input_dir = "../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/"
retail_centres = gpd.read_parquet(input_dir + "3_afterfill.parquet")
#normalize the data

from sklearn.preprocessing import normalize

# Normalize POI vectors to unit length (row-wise L2 norm)
poi_cols = ['Dining and Drinking' , 'Leisure & Travel', 'Community & Health', 'Commerce']
retail_centres[poi_cols] = retail_centres[poi_cols].fillna(0)
# poi_matrix = normalize(retail_centres[poi_cols].values, norm='l2', axis=1)
poi_matrix = retail_centres[poi_cols].values
retail_centres['poi_vec'] = poi_matrix.tolist()


import geopandas as gpd
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.cluster import AgglomerativeClustering
import warnings
from scipy.sparse import csr_matrix


# Parameters
alphas = [0, 0.2, 0.4, 0.5, 0.6, 0.8, 1]
clus_sizes= [50,75,100,125,150]
SEED = 42
np.random.seed(SEED)

k_by_network_size = lambda n,clus_size: max(1, max(1, n // clus_size))  # heuristic for # clusters

all_clustered = []
network_numbers = retail_centres['network_number'].value_counts().index.tolist()

for network_number in tqdm(network_numbers):
    rc = retail_centres[retail_centres['network_number'] == network_number].copy()

    if len(rc) < 2:
        for clus_size in clus_sizes:
            rc[f'subcluster_agglom_c{clus_size}_ward'] = 0
        all_clustered.append(rc)
        continue

    h3_list = rc['h3_11'].tolist()
    h3_index = {h: i for i, h in enumerate(h3_list)}
    N = len(h3_list)

    # Connectivity matrix from H3 neighbors
    rows, cols = [], []
    for h in h3_list:
        neighbors = h3.grid_ring(h, 1)
        i = h3_index[h]
        for n in neighbors:
            if n in h3_index:
                j = h3_index[n]
                rows.append(i)
                cols.append(j)
    data = np.ones(len(rows), dtype=int)
    connectivity = csr_matrix((data, (rows, cols)), shape=(N, N))

    poi_matrix = np.stack(rc['poi_vec'].values)  # shape (N, 4), unnormalized

    for clus_size in clus_sizes:
        k = k_by_network_size(N, clus_size)

        try:
            model = AgglomerativeClustering(
                n_clusters=k,
                linkage='ward',
                connectivity=connectivity
            )
            labels = model.fit_predict(poi_matrix)
            rc[f'subcluster_agglom_c{clus_size}_ward'] = labels
        except Exception as e:
            warnings.warn(f"Clustering failed for network {network_number}, clus_size {clus_size}: {e}")
            rc[f'subcluster_agglom_c{clus_size}_ward'] = 0

    all_clustered.append(rc)

# Combine all
retail_centres_clustered = pd.concat(all_clustered, ignore_index=True)
retail_centres_clustered.to_parquet(input_dir + "4_subclustered_agglom_multi_resolution_ward.parquet", index=False)

100%|██████████| 12044/12044 [01:18<00:00, 153.39it/s]


In [ ]:
alphas = [0,0.2,0.4,0.5,0.6,0.8,1]  # Density to POI weighting
# resolutions = [0.01, 0.05, 0.08, 0.1, 0.12, 0.15, 0.3]
resolutions = [0.05,0.1,0.15,0.3]  # Resolution parameters for clustering
all_clustered = []

network_numbers = retail_centres['network_number'].value_counts().index.tolist()

for network_number in tqdm(network_numbers):
    rc = retail_centres[retail_centres['network_number'] == network_number].copy()

    if len(rc) < 2:
        # Not enough to cluster
        for alpha in alphas:
            for res in resolutions:
                rc[f'subcluster_louvain_r{res}_a{alpha}'] = 0
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = 0
        all_clustered.append(rc)
        continue

    # Build lookup dictionaries once
    poi_lookup = dict(zip(rc['h3_11'], rc['poi_vec']))
    density_lookup = dict(zip(rc['h3_11'], rc['foursquare_count']))
    h3_set = set(rc['h3_11'])

    for alpha in alphas:
        # Build graph
        G = nx.Graph()
        # print(f"Building graph for network number {network_number} with alpha {alpha}")
        for h in sorted(h3_set):
            G.add_node(h, density=density_lookup[h])
            neighbors = h3.grid_ring(h, 1)
            for n in sorted(neighbors):
                if n in h3_set and not G.has_edge(h, n):
                    sim = np.dot(poi_lookup[h], poi_lookup[n])
                    d1, d2 = density_lookup[h], density_lookup[n]
                    density_weight = 1 / (1 + abs(d1 - d2))
                    weight = alpha * sim + (1 - alpha) * density_weight

                    if weight > 0:
                        G.add_edge(h, n, weight=weight)
        # Prepare igraph
        h3_nodes = list(G.nodes)
        h3_id_map = {h: i for i, h in enumerate(h3_nodes)}
        edges = [(h3_id_map[u], h3_id_map[v]) for u, v in G.edges()]
        weights = [G[u][v]['weight'] for u, v in G.edges()]

        if edges:
            G_ig = ig.Graph(n=len(h3_nodes), edges=edges, directed=False)

            G_ig.es['weight'] = weights
        else:
            G_ig = None

        for res in resolutions:
            # Louvain
            # print(f"Running Louvain with alpha {alpha} and resolution {res}")
            # communities = nx.community.louvain_communities(G, weight='weight', resolution=res, seed=seed)
            # h3_to_louvain = {node: i for i, community in enumerate(communities) for node in community}
            # rc[f'subcluster_louvain_r{res}_a{alpha}'] = rc['h3_11'].map(h3_to_louvain)
            # print(f"Louvain clustering completed for alpha {alpha} and resolution {res}")
            # Leiden
            if G_ig:
                partition_leiden = leidenalg.find_partition(
                    G_ig,
                    leidenalg.RBConfigurationVertexPartition,
                    weights=G_ig.es['weight'],
                    resolution_parameter=res,
                    seed=seed,
                    n_iterations=-1
                )
                membership = partition_leiden.membership
                h3_to_leiden = {h3_nodes[i]: membership[i] for i in range(len(h3_nodes))}
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = rc['h3_11'].map(h3_to_leiden)
            else:
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = 0

    all_clustered.append(rc)

# Combine all results
retail_centres_clustered = pd.concat(all_clustered, ignore_index=True)
retail_centres_clustered.to_parquet(input_dir + "4_subclustered_multi_resolution_alphas.parquet", index=False)


  8%|▊         | 945/12044 [02:11<12:42, 14.56it/s] 

In [19]:
retail_centres_clustered

,h3_11,network_number,foursquare_count,Dining and Drinking,Leisure & Travel,Community & Health,Commerce,top_category,dominant_type,retail_landuse,...,total_foursquare_count,trackNodes,geometry,poi_vec,subcluster_leiden_r0.1_a0,subcluster_leiden_r0.1_a0.5,subcluster_leiden_r0.1_a1,subcluster_louvain_r0.1_a0,subcluster_louvain_r0.1_a0.5,subcluster_louvain_r0.1_a1
0,8b195da49442fff,33946,2,0.0,0.0,0.0,2.0,[Commerce],Commerce,False,...,38053,5387,"POLYGON ((528310.386 180570.629, 528290.045 18...","[0.0, 0.0, 0.0]",6,7,632,NaN,NaN,NaN
1,8b194ad347a1fff,33946,4,1.0,0.0,0.0,3.0,[Commerce],Commerce,True,...,38053,5387,"POLYGON ((533582.222 182911.838, 533561.9 1828...","[1.0, 0.0, 0.0]",3,6,3,NaN,NaN,NaN
2,8b194ad3666efff,33946,4,2.0,0.0,0.0,2.0,"[Dining and Drinking, Commerce]",Commerce,False,...,38053,5387,"POLYGON ((531754.321 182096.407, 531733.993 18...","[1.0, 0.0, 0.0]",5,4,6,NaN,NaN,NaN
3,8b195da4d632fff,33946,17,7.0,2.0,0.0,8.0,[Commerce],mixed,True,...,38053,5387,"POLYGON ((529121.351 182255.838, 529101.014 18...","[0.9615239476408232, 0.27472112789737807, 0.0]",10,10,2,NaN,NaN,NaN
4,8b195da4d15bfff,33946,2,1.0,1.0,0.0,0.0,"[Dining and Drinking, Leisure & Travel]",Dining and Drinking,False,...,38053,5387,"POLYGON ((529974.836 182678.664, 529954.503 18...","[0.7071067811865475, 0.7071067811865475, 0.0]",8,5,9,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290134,8b195a8f1c51fff,67015,11,1.0,0.0,0.0,10.0,[Commerce],Commerce,False,...,11,1,"POLYGON ((270216.101 212037.043, 270195.004 21...","[1.0, 0.0, 0.0]",0,0,0,0.0,0.0,0.0
290135,8b1909a43d4dfff,3607,10,4.0,1.0,2.0,3.0,[Dining and Drinking],mixed,False,...,10,1,"POLYGON ((271531.989 665251.389, 271511.32 665...","[0.8728715609439696, 0.2182178902359924, 0.436...",0,0,0,0.0,0.0,0.0
290136,8b1909a480e8fff,3624,11,2.0,2.0,0.0,7.0,[Commerce],Commerce,False,...,11,1,"POLYGON ((274388.839 660374.072, 274368.174 66...","[0.7071067811865475, 0.7071067811865475, 0.0]",0,0,0,0.0,0.0,0.0
290137,8b1951ab0956fff,48696,11,7.0,2.0,1.0,1.0,[Dining and Drinking],Dining and Drinking,False,...,11,1,"POLYGON ((351657.797 395572.803, 351637.108 39...","[0.9525793444156804, 0.2721655269759087, 0.136...",0,0,0,0.0,0.0,0.0


In [111]:
#load the data and make a filtered version with only the top 100 networks
retail_centres_clustered = gpd.read_parquet(input_dir + "4_subclustered_multi_resolution.parquet")

In [113]:
#rename the network_numbers by size
network_numbers = retail_centres_clustered['network_number'].value_counts().index.tolist()
network_numbers = {k: i for i, k in enumerate(network_numbers)}
retail_centres_clustered['network_number'] = retail_centres_clustered['network_number'].map(network_numbers)
# Filter to only the top 100 networks
top_networks = retail_centres_clustered['network_number'].value_counts().nlargest(100).index.tolist()
retail_centres_top = retail_centres_clustered[retail_centres_clustered['network_number'].isin(top_networks)].copy()
#save the filtered data
retail_centres_top.to_parquet(input_dir + "4_subclustered_multi_resolution_top100.parquet", index=False)